In [0]:
from pyspark.sql import functions as F
import re

spark.sql("CREATE CATALOG IF NOT EXISTS company_ro")
spark.sql("CREATE SCHEMA IF NOT EXISTS company_ro.bronze")
spark.sql("""
CREATE TABLE IF NOT EXISTS company_ro.bronze.metadata_files_processed (
  file_path STRING, table_name STRING, ingested_at TIMESTAMP, row_count BIGINT, file_size BIGINT
) USING DELTA
""")
print("✓ Checkpoint ready")

In [0]:
bucket = "s3://ro-company-lake"
onrc_nomenclatoare_path = f"{bucket}/raw_v2/onrc/nomenclatoare/"

display(dbutils.fs.ls(onrc_nomenclatoare_path))

In [0]:
def list_all_files(path):
    all_files = []

    def recurse(p):
        for item in dbutils.fs.ls(p):
            if item.isDir():
                recurse(item.path)
            else:
                all_files.append(item.path)

    recurse(path)
    return all_files


all_files = list_all_files(onrc_nomenclatoare_path)

n_caen_files = [
    f for f in all_files
    if f.lower().endswith("/n_caen.csv") or f.lower().endswith("n_caen.csv")
]

print("n_caen files found:")
for f in n_caen_files:
    print(f)

print("Count:", len(n_caen_files))

if len(n_caen_files) == 0:
    raise ValueError("No n_caen.csv files found in S3.")

In [0]:
# Get already-processed files (serverless-compatible)
processed_rows = spark.table("company_ro.bronze.metadata_files_processed").filter(F.col("table_name") == "n_caen_raw").select("file_path").collect()
processed_files = [row["file_path"] for row in processed_rows]

# Filter to only NEW files
new_files = [f for f in n_caen_files if f not in processed_files]

if not new_files:
    print("\n✓ No new files to process.")
    dbutils.notebook.exit("No new data")

n_caen_path = new_files[0]
print("Using:", n_caen_path)

n_caen_raw = spark.read.option("header", True).option("inferSchema", False).option("sep", "^").csv(n_caen_path)

display(n_caen_raw.limit(50))
row_count = n_caen_raw.count()
print("Rows:", row_count)
print("Columns:", n_caen_raw.columns)

In [0]:
(
    n_caen_raw
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("company_ro.bronze.n_caen_raw")
)

print(f"✓ Appended {row_count:,} rows")

# Record in checkpoint
file_size = dbutils.fs.ls(n_caen_path)[0].size
checkpoint_df = (
    spark.createDataFrame(
        [(n_caen_path, "n_caen_raw", row_count, file_size)],
        ["file_path", "table_name", "row_count", "file_size"]
    )
    .withColumn("ingested_at", F.current_timestamp())
)
checkpoint_df.write.format("delta").mode("append").saveAsTable("company_ro.bronze.metadata_files_processed")
print("✓ Recorded file in checkpoint")

In [0]:
display(spark.sql("""
SELECT *
FROM company_ro.bronze.n_caen_raw
LIMIT 100
"""))

print(spark.table("company_ro.bronze.n_caen_raw").columns)